In [1]:
import ast
import os
import sys
import numpy as np
import pandas as pd
from tqdm import tqdm
import pickle
from shutil import copyfile
import random
random.seed(86)

In [2]:
#import qgrid
#import ipywidgets as widgets
#from ipywidgets import interact, interact_manual
# from IPython.display import display, Markdown, clear_output
# from ipywidgets import Layout, Button, Box, VBox
# from IPython.core.display import display, HTML

In [3]:
# data_dir = '../data/'
# assert os.path.exists(data_dir)

# paths = []
# for f in os.listdir(data_dir):
#     paths.append(os.path.join(data_dir, f))
# paths = sorted(paths)

In [4]:
print("Generating final tuples with everything...")
table_dir = '../data/'
# assert os.path.exists(table_dir)
### update glass ids in all (glass id, prop name, value, unit) tuples under the key table["prop_tuples"]
all_train_data = pickle.load(open(os.path.join(table_dir, 'all_train_data_pre_tuple_with_ids.pkl'), 'rb'))

Generating final tuples with everything...


In [5]:
# all_train_data[0]

In [6]:
# 510 density, 1140 Tg, 2010 ri, 2051 abbe
prop_ids = [510, 1140, 2010, 2051, 540, 50, 180, 60, 160, 1014, 1015, 3012, 3174, 1116, 1119, 1020, 1011, 1118, 70, 1306]
prop_names = ['Density', 'Glass transition temperature', 'Refractive index', 'Abbe value', "Young's modulus", 'Shear modulus', 'Vickers hardness', 'Poisson ratio', 'Fracture toughness', 'Crystallization temp', 'Melting temp', 'Electric conductivity', 'Dielectric constant', 'Softening Point (Temperature)', 'Annealing Point (Temperature)', 'Thermal expansion coefficient', 'Liquidus temperature', 'Softening Point (Viscosity)', 'Bulk modulus', 'Activation energy']
prop_ids_to_index = {prop_id: index for index, prop_id in enumerate(prop_ids)}
index_to_prop_ids = {index: prop_id for prop_id, index in enumerate(prop_ids)}
assert len(prop_ids)==len(prop_names)

all_train_data_new = []
count = set()
for idx, p in enumerate(all_train_data):
    # if os.path.exists(os.path.join(p, 'new.pkl')):
    table = all_train_data[idx]
    r, c = table['num_rows'], table['num_cols']
    pii, tid = table['pii'], table['t_idx']
    #print(pii+'__'+str(tid))
    if table['prop_table']: 
        table['prop_tuples'] = []
        for pi, prop_name in enumerate(prop_names):
            table['prop_tuples'].append([[[] for j1 in range(c)] for i1 in range (r)])
#             if prop_name in table['prop_names']:
            single_prop_table = table['props'][pi]
            if table['prop_orient'] =='col':
                try:
                    gid_index = table['col_label'].index(3)
                except ValueError:
                    gid_index = None
                prop_col_annotation = 4 + pi
                for j in range(c):
                    if table['col_label'][j] == prop_col_annotation:
                        for i in range(r):
                            if len(single_prop_table[i][j]) != 0:
                                old_tuple = single_prop_table[i][j][0]
                                if gid_index == None:
                                    gid = f'{pii}_{tid}_{i}_{j}'
                                else:
                                    gid = f'{pii}_{tid}_{i}_{j}_{table["act_table"][i][gid_index]}'
                                new_tuple = (gid, prop_name, old_tuple[1], old_tuple[2])
                                # print(new_tuple)
                                table['prop_tuples'][-1][i][j].append(new_tuple)
            elif table['prop_orient'] =='row':
                try:
                    gid_index = table['col_label'].index(3)
                except ValueError:
                    gid_index = None
                prop_row_annotation = 4 + pi
                for i in range(r):
                    if table['row_label'][i] == prop_row_annotation:
                        for j in range(c):
                            if len(single_prop_table[i][j]) != 0:
                                old_tuple = single_prop_table[i][j][0]
                                if gid_index == None:
                                    gid = f'{pii}_{tid}_{i}_{j}'
                                else:
                                    gid = f'{pii}_{tid}_{i}_{j}_{table["act_table"][gid_index][j]}'
                                new_tuple = (gid, prop_name, old_tuple[1], old_tuple[2])
                                table['prop_tuples'][-1][i][j].append(new_tuple)
            else:
                # if table['glass_id_type'] == 1:
                #     print(f'ERROR: glass ids parallel to prop orientation not found {table["pii"]} {table["t_idx"]}. skipping table...')
                assert False # should not occur          
        #pickle.dump(table, open(os.path.join(data_dir, pii+'__'+str(tid), 'new_with_tuples.pkl'), 'wb'))
        count.add(pii+'__'+str(tid))
    all_train_data_new.append(table)
    # else:
    #     print(f'ERROR: new.pkl not found for {table["pii"]} {table["t_idx"]}. skipping table...')
    #     print(p)

random.shuffle(all_train_data_new)
pickle.dump(all_train_data_new, open(os.path.join(table_dir, 'final_train_data_with_id_with_tuple.pkl'), 'wb'))
print(len(count))
print(len(all_train_data_new))

1021
2009
